# YOLO26 본 학습 (Colab)

데이터는 로컬에서 만든 yolo.zip (재원님 라벨 수정 + 시드 410 분할 적용).
ultralytics 버전은 로컬이랑 똑같이 8.4.183.

In [ ]:
# 로컬이랑 같은 버전으로 설치
!pip install -q ultralytics==8.4.183
!pip install -q -U kaggle

import torch
import ultralytics

print("ultralytics:", ultralytics.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

## W&B 로그인

키는 Colab 보안 비밀(WANDB_API_KEY)에서 읽음. 노트북엔 안 남음.
팀: codeit-prj1-team2 / 프로젝트: pill-object-detection

In [ ]:
!pip install -q wandb

import wandb
from google.colab import userdata

wandb.login(key=userdata.get("WANDB_API_KEY"))  # 보안 비밀에서 키 읽기

## 1. 데이터 준비

드라이브의 yolo.zip을 Colab으로 풀어서 씀 (드라이브에서 바로 읽으면 느림).
윈도우 zip이라 경로 구분자 \ → / 로 바꿔서 풀고, data.yaml 경로도 Colab 기준으로 고침.

In [ ]:
# 드라이브 전체에서 yolo.zip 찾기
!find /content/drive/MyDrive -iname "yolo*.zip" 2>/dev/null

# 내 드라이브 바로 아래 목록
!ls /content/drive/MyDrive

In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile
import yaml

drive.mount("/content/drive")

ZIP_PATH = Path("/content/drive/MyDrive/pill data/yolo.zip")
YOLO_DIR = Path("/content/yolo")

# 윈도우 zip이라 \ 를 / 로 바꿔가며 하나씩 풀기
with zipfile.ZipFile(ZIP_PATH) as z:
    for info in z.infolist():
        name = info.filename.replace("\\", "/")
        if name.endswith("/"):
            continue  # 폴더 항목은 건너뜀
        out = YOLO_DIR / name
        out.parent.mkdir(parents=True, exist_ok=True)
        out.write_bytes(z.read(info))

# data.yaml 경로를 Colab 기준으로
cfg_path = YOLO_DIR / "data.yaml"
cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))
cfg["path"] = str(YOLO_DIR)
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")

for s in ["train", "val"]:
    print(s, "이미지:", len(list((YOLO_DIR / "images" / s).glob("*.png"))),
          "/ 라벨:", len(list((YOLO_DIR / "labels" / s).glob("*.txt"))))
print("클래스 수:", len(cfg["names"]))

In [ ]:
# 새 zip이 맞는지 박스 수로 확인 (수정 반영이면 771)
saved = sum(len(p.read_text().splitlines()) for p in (YOLO_DIR / "labels").rglob("*.txt"))
print("박스:", saved)
assert saved == 771, "예전 zip인 것 같음. 드라이브 yolo.zip 확인 필요"

## 1-2. test 사진 준비

드라이브 test_images.zip에서 풀어서 씀.
(처음엔 Kaggle API로 받았는데 전체 다운로드는 막혀 있고, 하나씩 받으면 요청 제한에 걸려서 30분 넘게 걸림. 그래서 한 번 받은 걸 드라이브에 저장해둠)

In [ ]:
import zipfile

TEST_ZIP = Path("/content/drive/MyDrive/pill data/test_images.zip")
TEST_IMG_DIR = Path("/content/test_images")

with zipfile.ZipFile(TEST_ZIP) as z:
    z.extractall("/content")  # zip 안이 test_images/*.png 라서 /content/test_images로 풀림

print("test 이미지:", len(list(TEST_IMG_DIR.glob("*.png"))))

## 2. 본 학습 (기준 실험, 라벨 수정 반영)

처음 돌린 건 재원님 수정이 yolo.zip에 안 들어가 있었음 (변환 셀이 수정 셀보다 앞에 있었음).
고친 zip(박스 771)으로 다시 돌림. 이름 끝에 _fix 붙여서 예전 run이랑 구분.
seed 0, 1, 2로 점수가 얼마나 흔들리는지도 같이 봄.

In [ ]:
import wandb
from ultralytics import YOLO, settings

settings.update({"wandb": True})

WANDB_ENTITY = "codeit-prj1-team2"
WANDB_PROJECT = "pill-object-detection"

MODEL = "yolo26s"
EPOCHS = 100
RUNS_DIR = Path("/content/drive/MyDrive/pill data/runs")

def train_one(seed, exp="", **aug):
    # exp: 실험 이름 (run 이름 끝에 붙음), aug: 기본값에서 바꿀 설정
    run_name = f"{MODEL}_e{EPOCHS}_s{seed}_fix" + (f"_{exp}" if exp else "")

    # 기본값 먼저, 실험에서 넘긴 값으로 덮어씀
    args = dict(
        data=str(YOLO_DIR / "data.yaml"),
        epochs=EPOCHS,
        patience=30,
        imgsz=640,
        batch=16,
        device=0,
        seed=seed,
        workers=2,
        project=str(RUNS_DIR),
        name=run_name,
        exist_ok=True,
    )
    args.update(aug)

    wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        name=f"cy_{run_name}",
        tags=["yolo26", "label-fixed"] + ([exp] if exp else ["baseline"]),
        config={
            "model": MODEL,
            "split_seed": 410,
            "data": "yolo.zip 수정판 (재원 라벨 수정 반영, 박스 771)",
            **{k: v for k, v in args.items() if k not in ("data", "project", "name")},
        },
    )

    model = YOLO(f"{MODEL}.pt")
    model.train(**args)
    return run_name

## 3. test 예측 → 제출 csv

학습한 모델로 바로 예측. csv는 run 폴더(드라이브)에 저장되고 내 컴퓨터로도 받아짐.
처음엔 s0_fix로 로컬(VS Code) 결과 4902행이랑 맞는지 확인.

In [ ]:
from pathlib import Path
from ultralytics import YOLO

RUNS_DIR = Path("/content/drive/MyDrive/pill data/runs")  # 학습 결과 폴더

import re
import pandas as pd
from google.colab import files

COLS = ["image_id", "category_id", "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score"]
N_TEST = len(list(TEST_IMG_DIR.glob("*.png")))

def predict_run(run_name, imgsz=640, conf=0.01, download=True):
    run_dir = RUNS_DIR / run_name
    weights = run_dir / "weights" / "best.pt"
    if not weights.exists():
        print(f"[건너뜀] {run_name}: best.pt 없음")
        return

    model = YOLO(weights)
    rows = []
    for r in model.predict(
        source=str(TEST_IMG_DIR),  # 폴더째 넘겨야 한 장씩 읽음
        imgsz=imgsz,               # 학습 때랑 같은 크기
        conf=conf,
        device=0,
        stream=True,
        verbose=False,
    ):
        image_id = int(re.findall(r"\d+", Path(r.path).stem)[-1])
        for (x1, y1, x2, y2), c, s in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist()):
            rows.append([
                image_id,
                int(model.names[int(c)]),  # 0~55 → 원래 약 번호
                round(x1, 2), round(y1, 2),
                round(x2 - x1, 2), round(y2 - y1, 2),
                round(s, 4),
            ])

    sub = pd.DataFrame(rows, columns=COLS)
    sub.insert(0, "annotation_id", range(1, len(sub) + 1))

    sub_path = run_dir / f"submission_{run_name}.csv"  # 이름에 run 넣어서 안 헷갈리게
    sub.to_csv(sub_path, index=False)

    n_img = sub["image_id"].nunique()
    note = "" if n_img == N_TEST else "  ← 예측 안 나온 이미지 있음"
    print(f"[완료] {run_name}: {len(sub)}행, 이미지 {n_img}/{N_TEST}{note}")

    if download:
        files.download(str(sub_path))  # 내 컴퓨터로 받기

## 3-1. Kaggle 제출

Colab에서 바로 제출. 키는 보안 비밀 KAGGLE_JSON.
점수는 채점에 1~2분 걸려서 check_scores()로 따로 봄.

In [ ]:
import json, os, subprocess
from google.colab import userdata

COMP = "ai15-level1-project"

def kaggle_login():
    # 보안 비밀의 kaggle.json을 파일로 (세션마다 새로 필요)
    cred = json.loads(userdata.get("KAGGLE_JSON"))
    d = Path.home() / ".kaggle"
    d.mkdir(exist_ok=True)
    (d / "kaggle.json").write_text(json.dumps(cred))
    os.chmod(d / "kaggle.json", 0o600)

def submit(run_name, msg):
    kaggle_login()
    path = RUNS_DIR / run_name / f"submission_{run_name}.csv"
    subprocess.run(["kaggle", "competitions", "submit", "-c", COMP,
                    "-f", str(path), "-m", f"{run_name} {msg}"], check=True)

def check_scores():
    kaggle_login()
    subprocess.run(["kaggle", "competitions", "submissions", "-c", COMP], check=True)

## 4. val–test 차이 원인 보기

test는 정답이 없어서, 사진마다 자신 있게(score 0.5 이상) 찾은 박스 수를 train 알약 수랑 비교함.
test에서 찾은 개수가 확 적으면 못 알아보는 알약이 있다는 뜻.

In [ ]:
RUN = "yolo26s_e100_s0_fix"
sub = pd.read_csv(RUNS_DIR / RUN / f"submission_{RUN}.csv")

# test: 사진마다 score 0.5 이상 박스 수 (하나도 없는 사진은 0)
test_ids = [int(p.stem) for p in TEST_IMG_DIR.glob("*.png")]
test_cnt = (sub[sub["score"] >= 0.5].groupby("image_id").size()
            .reindex(test_ids, fill_value=0))

# train+val: 사진마다 정답 박스 수 (yolo 라벨 줄 수)
gt_cnt = pd.Series([len(p.read_text().splitlines())
                    for p in (YOLO_DIR / "labels").rglob("*.txt")])

print("사진당 알약 수 (train+val 정답)")
print(gt_cnt.value_counts().sort_index().to_string())
print("\n사진당 자신 있게 찾은 수 (test 예측)")
print(test_cnt.value_counts().sort_index().to_string())

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

conf_sub = sub[sub["score"] >= 0.5]

few = test_cnt[test_cnt <= 2].index.tolist()   # 1~2개만 찾은 사진
many = test_cnt[test_cnt >= 5].index.tolist()  # 5개 찾은 사진

random.seed(0)  # 다시 돌려도 같은 사진 나오게
pick_few = random.sample(few, min(4, len(few)))
pick_many = random.sample(many, min(4, len(many)))
print("2개 이하:", pick_few, "/ 5개:", pick_many)

def show(ids, title):
    fig, axes = plt.subplots(1, len(ids), figsize=(5 * len(ids), 7), squeeze=False)
    for ax, iid in zip(axes[0], ids):
        ax.imshow(Image.open(TEST_IMG_DIR / f"{iid}.png"))
        ax.set_title(f"{iid}.png")
        ax.axis("off")
        for b in conf_sub[conf_sub["image_id"] == iid].itertuples():
            ax.add_patch(patches.Rectangle((b.bbox_x, b.bbox_y), b.bbox_w, b.bbox_h,
                                           fill=False, edgecolor="red", linewidth=2))
            ax.text(b.bbox_x, b.bbox_y - 8, f"{b.category_id} ({b.score:.2f})",
                    color="white", fontsize=9, backgroundcolor="red")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

show(pick_few, "2개 이하로 찾은 사진")
show(pick_many, "5개 찾은 사진")

In [ ]:
import itertools
from collections import Counter
import numpy as np
import yaml

# yolo 번호 → 원래 약 번호
names = yaml.safe_load((YOLO_DIR / "data.yaml").read_text(encoding="utf-8"))["names"]
idx2cat = {int(k): int(v) for k, v in names.items()}

def iou(a, b):  # a, b = (x, y, w, h)
    ax2, ay2, bx2, by2 = a[0] + a[2], a[1] + a[3], b[0] + b[2], b[1] + b[3]
    iw = max(0, min(ax2, bx2) - max(a[0], b[0]))
    ih = max(0, min(ay2, by2) - max(a[1], b[1]))
    inter = iw * ih
    return inter / (a[2] * a[3] + b[2] * b[3] - inter)

# 1. 같은 알약에 겹친 박스 찾기 (score 0.5 이상끼리)
rows = []
for iid, g in conf_sub.groupby("image_id"):
    bs = g[["bbox_x", "bbox_y", "bbox_w", "bbox_h", "category_id", "score"]].values
    for i, j in itertools.combinations(range(len(bs)), 2):
        v = iou(bs[i], bs[j])
        if v >= 0.5:
            rows.append([iid, int(bs[i][4]), round(bs[i][5], 2), int(bs[j][4]), round(bs[j][5], 2), round(v, 2)])
dup = pd.DataFrame(rows, columns=["image_id", "약A", "scoreA", "약B", "scoreB", "IoU"])

print("겹친 박스 있는 사진:", dup["image_id"].nunique(), "/", N_TEST)
print("\n[5개 사진에서 겹친 것]")
print(dup[dup["image_id"].isin(pick_many)].to_string(index=False))
print("\n[자주 헷갈리는 약 쌍 Top 10]")
print(dup.apply(lambda r: tuple(sorted((r["약A"], r["약B"]))), axis=1).value_counts().head(10).to_string())

# 2. train 박스 수 vs test 예측 수
train_cnt = Counter()
for p in (YOLO_DIR / "labels" / "train").glob("*.txt"):
    for line in p.read_text().splitlines():
        train_cnt[idx2cat[int(line.split()[0])]] += 1

tbl = pd.DataFrame({
    "train박스": pd.Series(train_cnt),
    "test예측": conf_sub["category_id"].value_counts(),
}).fillna(0).astype(int)
tbl["배율"] = (tbl["test예측"] / tbl["train박스"].replace(0, np.nan)).round(1)  # 분포 같으면 약 4.5

print("\n[그림에서 헷갈린 약들]")
print(tbl.loc[[c for c in [3832, 13395, 3743, 28763, 12778] if c in tbl.index]].to_string())
print("\n[배율 높은 약 Top 10 — 모르는 약을 떠안고 있을 후보]")
print(tbl.sort_values("배율", ascending=False).head(10).to_string())

## 5. 실험: 상하 뒤집기

기본은 좌우 뒤집기만 켜져 있음. 상하도 켜서 알약 방향을 4가지로.
회전은 안 넣음 (네모 박스라 돌리면 박스가 커져서 정밀 채점에 불리).
기준: s0_fix Kaggle 0.410, 시드 흔들림 0.02 → 0.43 넘어야 효과 있다고 봄.

In [ ]:
run_name = train_one(0, exp="flipud", flipud=0.5)  # 상하 뒤집기 50%
predict_run(run_name)

In [ ]:
submit("yolo26s_e100_s0_fix_flipud", "상하 뒤집기")

In [ ]:
check_scores()

### 결과: 상하 뒤집기

| run | val mAP50-95 | Kaggle |
|---|---|---|
| s0_fix / s1_fix / s2_fix (기준) | 0.975 / 0.963 / 0.956 | 0.410 / 0.410 / 0.390 |
| s0_fix_flipud | 0.968 | 0.403 |

- val, Kaggle 둘 다 기준 시드 범위 안 → **효과 없음**
- 문제는 비슷한 알약을 구분 못 하는 건데, 뒤집기는 방향만 다양하게 해줄 뿐이라 구분하는 데는 도움이 안 된 걸로 보임
- 다음 실험부터는 상하 뒤집기 빼고 기본값으로 감
- 이번부터 Colab에서 바로 제출함 (submit 함수)

## 6. 실험: 해상도 1280

640은 원본(976×1280)을 절반으로 줄인 것. 헷갈린 약들이 각인 글씨로 구분되는 경우가 많아서 원본 크기로 봄.
1280에서도 안 오르면 해상도는 원인이 아니라고 보고 960은 안 함.
메모리 때문에 batch 16 → 8. 예측도 1280으로 해야 함.
기준: Kaggle 0.43 이상이면 효과 있음.

In [ ]:
run_name = train_one(0, exp="img1280", imgsz=1280, batch=8)
predict_run(run_name, imgsz=1280, download=False)  # 학습 크기랑 맞춤
submit(run_name, "해상도 1280")

In [ ]:
check_scores()

### 결과: 해상도 1280

| run | val mAP50-95 | Kaggle |
|---|---|---|
| 기준 seed 0/1/2 | 0.975 / 0.963 / 0.956 | 0.410 / 0.410 / 0.390 |
| s0_fix_flipud | 0.968 | 0.403 |
| s0_fix_img1280 | 0.965 | 0.403 |

- 원본 크기까지 올려도 그대로 → 해상도는 원인 아님. 960은 안 함
- 증강, 해상도 둘 다 안 움직임 → 학습 방법보다 데이터 한계 (희귀 약 train 3장, test엔 훨씬 많이 나옴)
- 지금 데이터로는 0.40 근처가 천장으로 보임. 실험은 여기서 멈추고 외부 데이터 붙일 준비부터